# C6 — Validação Final (abertura única do teste)

## Objetivo

Abrir a partição de **teste** exatamente uma vez, após o gate `G9_holdout` estar aprovado, e produzir a avaliação final do modelo escolhido.

Este notebook:

- exige `G7_calibracao`, `G8_limiar` e `G9_holdout` **aprovados**;
- lê do `supervised.yaml` a **configuração congelada**: finalista, hiperparâmetros, método de calibração e limiar;
- reconstrói o pipeline deterministicamente via `build_finalista` da fábrica;
- treina no **treino**, calibra na **validação** e avalia **uma única vez** no **teste**;
- produz métricas agregadas, **IC por bootstrap**, **análise de erros (falsos negativos primeiro)**, matriz de confusão, métricas por ano, métricas por grupo e calibração por ano;
- usa `faixa_etaria` **derivada de `idade_anos`** quando a coluna não estiver persistida, e `sg_uf` (ou qualquer coluna de auditoria) **persistido** no artefato;
- ao final, **marca `G9_holdout` como `executado`** no `supervised.yaml` (edição cirúrgica, preservando comentários).

> A partir do momento em que este notebook roda, **nada mais pode ser alterado** no modelo, features, calibração ou limiar sem violar o contrato metodológico do projeto.

## 1. Bootstrap

In [ ]:
from pathlib import Path
import json
import os
import platform
import re
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def encontrar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src").is_dir() and (pasta / "configs").is_dir():
            return pasta
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("ROOT_DIR:", ROOT_DIR)
print("Python:", sys.version.split()[0])
print("Sistema:", platform.system(), platform.release())

In [ ]:
from src.utils import config as C
from src.utils import env
from src.utils.reproducibility import Run
from src.data.supervised_artifact import load_supervised_artifact
from src.evaluation.metrics import (
    metricas_completas,
    ic_bootstrap,
    auc,
    average_precision,
    brier as brier_metric,
)
from src.evaluation.errors import matriz_confusao, metricas_por_grupo
from src.evaluation.calibration import (
    curva_confiabilidade,
    ece,
    mce,
    brier_decomposicao,
)
from src.features.engineering import faixa_etaria as _faixa_etaria
from src.models.factory import build_finalista
from src.models.calibration import calibrar

try:
    import sklearn.frozen  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Este notebook requer scikit-learn >= 1.6 "
        "(sklearn.frozen.FrozenEstimator). "
        f"ImportError: {exc}"
    ) from exc

env.carregar_dotenv()
env.garantir_diretorios()

TABLES_DIR = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print(C.resumo())

## 2. Verificar gates `G7`, `G8` e `G9`

Antes de qualquer coisa, confirmamos que a decisão metodológica está congelada. `G9_holdout` precisa estar em `aprovado` **nesta entrada** — o notebook marcará `executado` ao final.

In [ ]:
GATES_EXIGIDOS = ["G7_calibracao", "G8_limiar", "G9_holdout"]

gates_cfg = {nome: (C.cfg("gates", nome) or {}) for nome in GATES_EXIGIDOS}
status_gates = {nome: cfg.get("status") for nome, cfg in gates_cfg.items()}

display(pd.DataFrame(
    {"gate": list(status_gates.keys()), "status": list(status_gates.values())}
))

gates_faltando = [
    nome for nome, st in status_gates.items() if st != "aprovado"
]
if gates_faltando:
    raise RuntimeError(
        "Os gates abaixo precisam estar 'aprovado' antes de abrir o teste: "
        f"{gates_faltando}. Edite configs/supervised.yaml e rerode."
    )

n_acessos_antes = int((gates_cfg["G9_holdout"] or {}).get("n_acessos") or 0)
print("G7, G8 e G9 aprovados. O teste pode ser aberto uma única vez.")
print(f"n_acessos atual em G9_holdout: {n_acessos_antes}")

## 3. Carregar o artefato supervisionado canônico

In [ ]:
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "provisional").lower()

df, manifest = load_supervised_artifact(
    ROOT_DIR,
    expected_kind=COHORT_KIND,
)

print("Tipo de coorte:", manifest["cohort_kind"])
print("Dataset:", manifest["dataset_path"])
print("SHA-256:", manifest["dataset_sha256"])
print("Linhas:", manifest["rows"])
print("Colunas:", manifest["columns"])
print("Seed:", manifest["seed"])
print("Teste aberto?", manifest["test_opened"])
print("Partições:", manifest["partition_counts"])
print("Colunas de auditoria:", manifest.get("audit_columns") or [])

## 4. Guarda contra reabertura do teste

Procura por manifestos anteriores do C6 apontando para o mesmo `dataset_sha256`. Se encontrar, **bloqueia** a execução — a menos que `SRAG_ALLOW_TEST_REOPEN=1` seja definida explicitamente (por exemplo, em uma recontagem por bug comprovado).

In [ ]:
ALLOW_REOPEN = os.getenv("SRAG_ALLOW_TEST_REOPEN", "0") == "1"

manifestos_c6_anteriores = []
for caminho in TABLES_DIR.glob("C6_validacao_final_*_manifesto.json"):
    try:
        dados = json.loads(caminho.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        continue
    if dados.get("dataset_sha256") == manifest["dataset_sha256"]:
        manifestos_c6_anteriores.append({
            "arquivo": str(caminho),
            "teste_aberto_em_utc": dados.get("teste_aberto_em_utc"),
        })

if manifestos_c6_anteriores and not ALLOW_REOPEN:
    display(pd.DataFrame(manifestos_c6_anteriores))
    raise RuntimeError(
        "O teste já foi aberto para este dataset_sha256. "
        "Reabrir o teste viola o contrato metodológico do projeto. "
        "Se for uma recontagem deliberada e justificada, defina "
        "SRAG_ALLOW_TEST_REOPEN=1 e registre o motivo no docs/decisoes.md."
    )

if manifestos_c6_anteriores and ALLOW_REOPEN:
    print("ATENÇÃO: reabertura explícita autorizada por SRAG_ALLOW_TEST_REOPEN=1.")
    print("Registre o motivo e a data em docs/decisoes.md.")
else:
    print("Nenhuma abertura anterior deste dataset foi encontrada. Prosseguindo.")

## 5. Ler a configuração congelada do `supervised.yaml`

O artefato congelado é a **configuração** do YAML, não um arquivo `joblib`. O pipeline é reconstruído deterministicamente a partir dela:

- `selecao_modelos.finalista_escolhido`;
- hiperparâmetros do finalista em `models.logistic` ou `models.boosting`;
- `calibracao.metodo` (escolhido no C5);
- `limiar.otimo` e `limiar.metrica`;
- `features.final` (obrigatório nesta etapa).

In [ ]:
selecao_cfg = C.cfg("selecao_modelos") or {}
finalista_escolhido = selecao_cfg.get("finalista_escolhido")
finalistas_declarados = list(selecao_cfg.get("finalistas") or [])

if finalista_escolhido is None:
    if len(finalistas_declarados) == 1:
        finalista_escolhido = finalistas_declarados[0]
    else:
        raise RuntimeError(
            "`selecao_modelos.finalista_escolhido` não está definido e há "
            f"múltiplos finalistas ({finalistas_declarados}). "
            "Defina explicitamente o finalista único em supervised.yaml "
            "antes de rodar o C6."
        )

calib_cfg = C.cfg("calibracao") or {}
calib_metodo = calib_cfg.get("metodo")
calib_ativa = bool(calib_cfg.get("ativa", True))
calib_fonte = str(calib_cfg.get("fonte") or "validacao")

limiar_cfg = C.cfg("limiar") or {}
limiar_final = limiar_cfg.get("otimo")
limiar_metrica = limiar_cfg.get("metrica")

features_final = C.cfg("features", "final")

if features_final is None:
    raise RuntimeError(
        "`features.final` está vazio. O teste não pode ser aberto sem "
        "a lista congelada de features. Preencha no supervised.yaml e rerode."
    )
if calib_ativa and calib_metodo is None:
    raise RuntimeError(
        "`calibracao.ativa: true`, mas `calibracao.metodo` está vazio. "
        "Escolha o método no YAML (resultado do C5) e rerode."
    )
if limiar_final is None:
    raise RuntimeError(
        "`limiar.otimo` está vazio. Escolha o limiar no C5, registre no YAML "
        "e rerode."
    )
if calib_fonte != "validacao":
    raise RuntimeError(
        f"`calibracao.fonte` deve ser 'validacao' (valor atual: {calib_fonte!r}). "
        "A calibração não pode tocar no teste."
    )

config_congelada = pd.DataFrame([
    {"chave": "finalista_escolhido", "valor": finalista_escolhido},
    {"chave": "calibracao.ativa", "valor": calib_ativa},
    {"chave": "calibracao.metodo", "valor": calib_metodo},
    {"chave": "calibracao.fonte", "valor": calib_fonte},
    {"chave": "limiar.otimo", "valor": limiar_final},
    {"chave": "limiar.metrica", "valor": limiar_metrica},
    {"chave": "features.final (n)", "valor": len(features_final)},
])

display(config_congelada)

## 6. Validar contrato e separar treino/validação/teste

A validação é idêntica à dos notebooks anteriores. É a **primeira vez** que a variável `teste` é materializada.

In [ ]:
PARTICAO = manifest["partition_column"]
ALVO = manifest["target_column"]
FEATURES = list(features_final)

particoes_observadas = set(
    df[PARTICAO].dropna().astype(str).unique().tolist()
)

alvo_binario = (
    pd.to_numeric(df[ALVO], errors="coerce")
    .dropna()
    .astype(int)
    .isin([0, 1])
    .all()
)

verificacoes = {
    "artefato_nao_vazio": not df.empty,
    "particao_presente": PARTICAO in df.columns,
    "alvo_presente": ALVO in df.columns,
    "features_presentes": set(FEATURES).issubset(df.columns),
    "particoes_esperadas": particoes_observadas == {"treino", "validacao", "teste"},
    "alvo_binario": bool(alvo_binario),
    "teste_fechado_no_manifesto_antes_desta_execucao": manifest.get("test_opened") is False,
}

display(pd.DataFrame({
    "verificacao": list(verificacoes.keys()),
    "resultado": list(verificacoes.values()),
}))

if not all(verificacoes.values()):
    falhas = [k for k, v in verificacoes.items() if not v]
    raise RuntimeError(f"Falhas no contrato do artefato: {falhas}")

proibidas = set(C.cfg("features", "proibidas") or [])
if set(FEATURES) & proibidas:
    raise RuntimeError(
        f"features.final contém colunas proibidas: {sorted(set(FEATURES) & proibidas)}"
    )

In [ ]:
treino = df.loc[df[PARTICAO] == "treino"].copy()
validacao = df.loc[df[PARTICAO] == "validacao"].copy()
teste = df.loc[df[PARTICAO] == "teste"].copy()

X_train = treino[FEATURES].copy()
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8")

X_val = validacao[FEATURES].copy()
y_val = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8")

X_test = teste[FEATURES].copy()
y_test = pd.to_numeric(teste[ALVO], errors="coerce").astype("int8")

assert X_train.index.equals(y_train.index)
assert X_val.index.equals(y_val.index)
assert X_test.index.equals(y_test.index)

display(pd.DataFrame({
    "particao": ["treino", "validacao", "teste"],
    "n": [len(treino), len(validacao), len(teste)],
    "positivos": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "prevalencia": [float(y_train.mean()), float(y_val.mean()), float(y_test.mean())],
}))

print("Teste materializado agora. Esta execução abre o teste uma única vez.")

## 7. Reconstruir o pipeline a partir da configuração congelada

Treinamos no **treino** e, se `calibracao.ativa` for `true`, ajustamos o calibrador na **validação** com o método congelado. O teste nunca é usado para ajustar nada.

A construção do pipeline é delegada a `build_finalista` (`src.models.factory`) — mesma fábrica usada no C5.

In [ ]:
# Mapeamento do nome persistido no YAML para o nome aceito pela fábrica.
ALIASES_FINALISTA = {
    "logistic":               "logistic",
    "boosting":               "boosting",
    "hist_gradient_boosting": "boosting",
}

if finalista_escolhido not in ALIASES_FINALISTA:
    raise ValueError(
        f"Finalista desconhecido: {finalista_escolhido!r}. "
        f"Suportados: {sorted(ALIASES_FINALISTA)}."
    )


def construir_finalista(nome_yaml: str):
    """
    Constrói o pipeline via `build_finalista`, passando os hiperparâmetros
    congelados do YAML como overrides. Valida que os campos críticos estão
    preenchidos — o modelo precisa estar congelado antes do C6.
    """
    nome_factory = ALIASES_FINALISTA[nome_yaml]

    if nome_factory == "logistic":
        cfg = C.cfg("models", "logistic") or {}
        c_val = cfg.get("C")
        if c_val is None:
            raise RuntimeError(
                "`models.logistic.C` está vazio no YAML. O modelo final "
                "precisa estar congelado antes do C6."
            )
        return build_finalista(
            "logistic",
            C=float(c_val),
            solver=str(cfg.get("solver") or "lbfgs"),
            max_iter=int(cfg.get("max_iter") or 1000),
            class_weight=cfg.get("class_weight") or "balanced",
        )

    # nome_factory == "boosting"
    cfg = C.cfg("models", "boosting") or {}
    lr = cfg.get("learning_rate")
    mi = cfg.get("max_iter")
    if lr is None or mi is None:
        raise RuntimeError(
            "`models.boosting.learning_rate` / `max_iter` precisam estar "
            "preenchidos antes do C6 (resultado do C4)."
        )
    return build_finalista(
        "boosting",
        learning_rate=float(lr),
        max_iter=int(mi),
        max_depth=(None if cfg.get("max_depth") is None else int(cfg["max_depth"])),
        max_leaf_nodes=int(cfg.get("max_leaf_nodes") or 31),
        min_samples_leaf=int(cfg.get("min_samples_leaf") or 20),
        l2_regularization=float(cfg.get("l2_regularization") or 0.0),
        early_stopping=bool(cfg.get("early_stopping") or False),
        class_weight=cfg.get("class_weight") or "balanced",
    )

In [ ]:
nome_finalista = finalista_escolhido
if nome_finalista is None:
    raise RuntimeError("`finalista_escolhido` não foi definido no YAML antes de construir o pipeline.")

nome_finalista = str(nome_finalista)
if nome_finalista not in ALIASES_FINALISTA:
    raise ValueError(
        f"Finalista desconhecido: {nome_finalista!r}. "
        f"Suportados: {sorted(ALIASES_FINALISTA)}."
    )

print(
    f"Construindo pipeline base: {nome_finalista} "
    f"(fábrica: {ALIASES_FINALISTA[nome_finalista]})"
)

pipe_base = construir_finalista(nome_finalista)
pipe_base.fit(X_train, y_train)

prob_val_base = pipe_base.predict_proba(X_val)[:, 1]
print(f"Brier base (validação): {np.mean((prob_val_base - y_val.to_numpy()) ** 2):.6f}")

if calib_ativa:
    print(f"Ajustando calibrador na validação: method={calib_metodo!r}")
    metodo_calibracao = str(calib_metodo).strip() if calib_metodo is not None else ""
    if not metodo_calibracao:
        raise RuntimeError(
            "`calibracao.metodo` está vazio no YAML. "
            "Defina um método de calibração antes de ajustar o calibrador."
        )
    calibrador = calibrar(pipe_base, X_val, y_val, method=metodo_calibracao)
    prob_val_final = calibrador.predict_proba(X_val)[:, 1]
    print(f"Brier calibrado (validação): {np.mean((prob_val_final - y_val.to_numpy()) ** 2):.6f}")
else:
    print("Calibração desativada — usando probabilidades brutas.")
    calibrador = None
    prob_val_final = prob_val_base

if not np.isfinite(prob_val_final).all() or not ((prob_val_final >= 0) & (prob_val_final <= 1)).all():
    raise RuntimeError("Probabilidades inválidas na validação após calibração.")

## 8. Abrir o teste — predição única

Este é o **único** ponto em todo o projeto em que a partição de teste é usada. Nada aqui é usado para reajustar o modelo, a calibração ou o limiar.

In [ ]:
abertura_teste_utc = pd.Timestamp.now(tz="UTC").isoformat()
print("Teste aberto em UTC:", abertura_teste_utc)

if calib_ativa:
    if calibrador is None:
        prob_test = pipe_base.predict_proba(X_test)[:, 1]
    else:
        prob_test = calibrador.predict_proba(X_test)[:, 1]
else:
    prob_test = pipe_base.predict_proba(X_test)[:, 1]

if not np.isfinite(prob_test).all():
    raise RuntimeError("Probabilidades não finitas no teste.")
if not ((prob_test >= 0) & (prob_test <= 1)).all():
    raise RuntimeError("Probabilidades fora de [0, 1] no teste.")

y_test_arr = y_test.to_numpy()
if limiar_final is None:
    raise RuntimeError("`limiar_final` não está definido no YAML. O teste não pode continuar.")

limiar_final_val = float(limiar_final)
y_pred_test = (prob_test >= limiar_final_val).astype(int)

print("Predições de teste concluídas.")
print("n teste:", len(prob_test))
print("Limiar aplicado:", float(limiar_final))

## 9. Métricas agregadas no teste

In [ ]:
if limiar_final is None:
    raise RuntimeError("`limiar_final` não está definido no YAML; não é possível calcular métricas no teste.")

if y_test_arr is None or prob_test is None:
    raise RuntimeError("`y_test_arr` e `prob_test` precisam estar definidos antes da avaliação final.")

limiar_final_val = float(limiar_final)

metricas_test = metricas_completas(
    np.asarray(y_test_arr, dtype=np.int8),
    np.asarray(prob_test, dtype=float),
    limiar=limiar_final_val,
)
metricas_df = pd.DataFrame([{
    "modelo": finalista_escolhido,
    "calibracao": (calib_metodo if calib_ativa else "nenhuma"),
    "limiar": float(limiar_final),
    **{k: float(v) for k, v in metricas_test.items()},
}])

display(metricas_df)

matriz_test = matriz_confusao(y_test_arr, y_pred_test)
display(matriz_test)

## 10. Intervalos de confiança por bootstrap

IC percentil para as métricas principais no teste, segundo `avaliacao.bootstrap` do `supervised.yaml`:

- `ap` (average precision) — métrica principal;
- `auc` (ROC-AUC);
- `brier`.

Bootstrap reamostra as linhas do teste com reposição. Amostras degeneradas (uma única classe) são descartadas internamente por `ic_bootstrap`.

In [ ]:
boot_cfg = C.cfg("avaliacao", "bootstrap") or {}
BOOT_ATIVO = bool(boot_cfg.get("ativo", True))
BOOT_REPS = int(boot_cfg.get("repeticoes") or 1000)
BOOT_NIVEL = float(boot_cfg.get("nivel") or 0.95)
BOOT_SEED = int(boot_cfg.get("seed") or C.SEED)
BOOT_ALPHA = 1.0 - BOOT_NIVEL

print(f"Bootstrap ativo: {BOOT_ATIVO} | reps={BOOT_REPS} | nível={BOOT_NIVEL} | seed={BOOT_SEED}")

ic_resultados = {}

if BOOT_ATIVO:
    for nome_metrica, fn in [
        ("ap",     average_precision),
        ("auc",    auc),
        ("brier",  brier_metric),
    ]:
        est, inf, sup = ic_bootstrap(
            y_test_arr,
            prob_test,
            metrica=fn,
            n_boot=BOOT_REPS,
            alpha=BOOT_ALPHA,
            seed=BOOT_SEED,
        )
        ic_resultados[nome_metrica] = {
            "estimativa": float(est),
            "ic_inferior": float(inf),
            "ic_superior": float(sup),
            "nivel": float(BOOT_NIVEL),
        }

ic_df = pd.DataFrame([
    {"metrica": k, **v} for k, v in ic_resultados.items()
])
display(ic_df)

## 11. Análise de erros — falsos negativos primeiro

`avaliacao.analise_erros.priorizar_falsos_negativos: true` no YAML. Os falsos negativos (óbitos previstos como cura) são o erro de maior custo clínico — por isso vêm primeiro.

Comparamos as distribuições de features numéricas entre FN e TP para enxergar onde o modelo erra mais.

In [ ]:
erros_cfg = C.cfg("avaliacao", "analise_erros") or {}
ERROS_ATIVA = bool(erros_cfg.get("ativa", True))
PRIORIZAR_FN = bool(erros_cfg.get("priorizar_falsos_negativos", True))

print(f"Análise de erros ativa: {ERROS_ATIVA} | priorizar FN: {PRIORIZAR_FN}")

# DataFrame alinhado por posição com y_test_arr / prob_test.
df_test_err = teste.reset_index(drop=True).copy()
df_test_err["y_true"] = y_test_arr
df_test_err["y_pred"] = y_pred_test
df_test_err["y_prob"] = prob_test

fn = df_test_err[(df_test_err["y_true"] == 1) & (df_test_err["y_pred"] == 0)]
tp = df_test_err[(df_test_err["y_true"] == 1) & (df_test_err["y_pred"] == 1)]
fp = df_test_err[(df_test_err["y_true"] == 0) & (df_test_err["y_pred"] == 1)]
tn = df_test_err[(df_test_err["y_true"] == 0) & (df_test_err["y_pred"] == 0)]

resumo_erros = pd.DataFrame([
    {"classe": "FN (óbito previsto cura)", "n": len(fn), "frac_do_total": len(fn) / len(df_test_err)},
    {"classe": "TP (óbito previsto óbito)", "n": len(tp), "frac_do_total": len(tp) / len(df_test_err)},
    {"classe": "FP (cura previsto óbito)", "n": len(fp), "frac_do_total": len(fp) / len(df_test_err)},
    {"classe": "TN (cura previsto cura)", "n": len(tn), "frac_do_total": len(tn) / len(df_test_err)},
])
display(resumo_erros)

if ERROS_ATIVA and len(fn) > 0 and len(tp) > 0:
    # Comparação de distribuições FN vs TP nas features numéricas.
    feats_num = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df_test_err[c])]
    linhas_cmp = []
    for c in feats_num:
        linhas_cmp.append({
            "feature": c,
            "fn_media":   float(fn[c].mean()),
            "tp_media":   float(tp[c].mean()),
            "fn_mediana": float(fn[c].median()),
            "tp_mediana": float(tp[c].median()),
        })
    comparacao_fn_tp = pd.DataFrame(linhas_cmp)
    comparacao_fn_tp["delta_media"] = comparacao_fn_tp["fn_media"] - comparacao_fn_tp["tp_media"]
    comparacao_fn_tp["abs_delta"] = comparacao_fn_tp["delta_media"].abs()
    comparacao_fn_tp = comparacao_fn_tp.sort_values("abs_delta", ascending=False).reset_index(drop=True)
    print("\nFeatures numéricas com maior diferença FN vs TP:")
    display(comparacao_fn_tp)
else:
    comparacao_fn_tp = pd.DataFrame()
    print("Sem FN ou sem TP suficientes para comparação de distribuições.")

# Amostra dos FN com maior probabilidade — onde o modelo chegou mais perto.
if len(fn) > 0:
    amostra_fn = (
        fn.sort_values("y_prob", ascending=False)
          .head(20)
          [ [c for c in FEATURES if c in fn.columns] + ["y_prob"] ]
    )
    print("\nTop 20 falsos negativos por probabilidade predita:")
    display(amostra_fn)
else:
    amostra_fn = pd.DataFrame()

## 12. Métricas por ano

In [ ]:
COLUNA_ANO = str(C.cfg("periodo", "coluna_ano") or "ano_internacao")

if COLUNA_ANO not in teste.columns:
    print(f"{COLUNA_ANO!r} ausente na partição de teste. Pulando métricas por ano.")
    metricas_por_ano_df = pd.DataFrame()
else:
    registros = []
    anos = sorted(teste[COLUNA_ANO].dropna().astype(int).unique().tolist())
    for ano in anos:
        mascara = teste[COLUNA_ANO].astype("Int64") == ano
        y_ano = y_test_arr[mascara.to_numpy()]
        p_ano = prob_test[mascara.to_numpy()]
        if len(y_ano) == 0:
            continue
        if len(np.unique(y_ano)) < 2:
            auc_ano = float("nan")
            ap_ano = float("nan")
        else:
            from sklearn.metrics import roc_auc_score, average_precision_score
            auc_ano = float(roc_auc_score(y_ano, p_ano))
            ap_ano = float(average_precision_score(y_ano, p_ano))
        brier_ano = float(np.mean((p_ano - y_ano) ** 2))
        ece_ano = float(ece(y_ano, p_ano, n_bins=10)) if len(y_ano) >= 20 else float("nan")
        registros.append({
            "ano_internacao": int(ano),
            "n": int(len(y_ano)),
            "positivos": int(y_ano.sum()),
            "prevalencia": float(y_ano.mean()),
            "ap": ap_ano,
            "auc": auc_ano,
            "brier": brier_ano,
            "ece": ece_ano,
        })
    metricas_por_ano_df = pd.DataFrame(registros)

display(metricas_por_ano_df)

## 13. Métricas por grupo

Grupos configurados em `avaliacao.grupos` do `supervised.yaml`.

Colunas tratadas com cuidado:

- **`faixa_etaria`** — se não estiver persistida no artefato, é **derivada de `idade_anos`** via `src.features.engineering.faixa_etaria`;
- **`sg_uf`** (e demais colunas de auditoria) — vêm do artefato via `manifest['audit_columns']`, persistidas pelo Notebook 03;
- qualquer grupo ausente é pulado com registro explícito.

In [ ]:
grupos_cfg = list(C.cfg("avaliacao", "grupos") or [])
min_n_grupo = int(C.cfg("avaliacao", "min_n_grupo") or 100)
audit_columns = list(manifest.get("audit_columns") or [])

print("Grupos configurados:", grupos_cfg)
print("min_n_grupo:", min_n_grupo)
print("Colunas de auditoria no manifesto:", audit_columns)

# df de teste alinhado por posição com y_test_arr / prob_test.
df_test_metricas = teste.reset_index(drop=True).copy()

# Deriva faixa_etaria se necessário e possível.
if "faixa_etaria" in grupos_cfg and "faixa_etaria" not in df_test_metricas.columns:
    if "idade_anos" in df_test_metricas.columns:
        df_test_metricas["faixa_etaria"] = _faixa_etaria(df_test_metricas["idade_anos"])
        print("faixa_etaria derivada de idade_anos para o cálculo dos grupos.")
    else:
        print(
            "faixa_etaria não persistida e idade_anos ausente — grupo será pulado. "
            "Persista uma delas via features.final ou audit_columns no Notebook 03."
        )

metricas_por_grupo_dict = {}
grupos_pulados = []

for grupo in grupos_cfg:
    if grupo not in df_test_metricas.columns:
        grupos_pulados.append(grupo)
        continue
    if limiar_final is None:
        raise RuntimeError(
            "`limiar.otimo` está vazio no YAML. "
            "Não é possível calcular métricas por grupo."
        )

    tabela = metricas_por_grupo(
        df_test_metricas,
        y_test_arr,
        prob_test,
        grupo,
        limiar=float(limiar_final),
        min_n=min_n_grupo,
    )
    metricas_por_grupo_dict[grupo] = tabela
    print(f"\n=== Grupo: {grupo} ===")
    display(tabela)

if grupos_pulados:
    print(f"\nGrupos ausentes na partição de teste (pulados): {grupos_pulados}")

## 14. Calibração por ano

In [ ]:
calib_por_ano_df = pd.DataFrame()
curvas_por_ano = {}

if COLUNA_ANO in teste.columns and not metricas_por_ano_df.empty:
    registros = []
    for ano in metricas_por_ano_df["ano_internacao"].tolist():
        mascara = teste[COLUNA_ANO].astype("Int64") == ano
        y_ano = y_test_arr[mascara.to_numpy()]
        p_ano = prob_test[mascara.to_numpy()]
        if len(y_ano) < 20 or len(np.unique(y_ano)) < 2:
            continue
        curva_ano = curva_confiabilidade(y_ano, p_ano, n_bins=10)
        decomp_ano = brier_decomposicao(y_ano, p_ano, n_bins=10)
        curvas_por_ano[int(ano)] = curva_ano
        registros.append({
            "ano_internacao": int(ano),
            "n": int(len(y_ano)),
            "ece": float(ece(y_ano, p_ano, n_bins=10)),
            "mce": float(mce(y_ano, p_ano, n_bins=10)),
            "brier": float(decomp_ano["brier"]),
            "reliability": float(decomp_ano["reliability"]),
            "resolution": float(decomp_ano["resolution"]),
            "uncertainty": float(decomp_ano["uncertainty"]),
        })
    calib_por_ano_df = pd.DataFrame(registros)

display(calib_por_ano_df)

## 15. Persistir resultados finais

In [ ]:
timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo = f"C6_validacao_final_{manifest['cohort_kind']}_{timestamp}"

caminho_metricas = TABLES_DIR / f"{prefixo}_metricas.csv"
caminho_matriz = TABLES_DIR / f"{prefixo}_matriz_confusao.csv"
caminho_por_ano = TABLES_DIR / f"{prefixo}_metricas_por_ano.csv"
caminho_calib_ano = TABLES_DIR / f"{prefixo}_calibracao_por_ano.csv"
caminho_manifesto = TABLES_DIR / f"{prefixo}_manifesto.json"
caminho_predicoes = PREDICTIONS_DIR / f"{prefixo}_predicoes_teste.parquet"
caminho_ic = TABLES_DIR / f"{prefixo}_ic_bootstrap.csv"
caminho_erros = TABLES_DIR / f"{prefixo}_analise_erros.csv"
caminho_fn_amostra = TABLES_DIR / f"{prefixo}_fn_amostra.csv"

metricas_df.to_csv(caminho_metricas, index=False, encoding="utf-8-sig")
matriz_test.to_csv(caminho_matriz, encoding="utf-8-sig")
metricas_por_ano_df.to_csv(caminho_por_ano, index=False, encoding="utf-8-sig")
calib_por_ano_df.to_csv(caminho_calib_ano, index=False, encoding="utf-8-sig")
if not ic_df.empty:
    ic_df.to_csv(caminho_ic, index=False, encoding="utf-8-sig")
if not comparacao_fn_tp.empty:
    comparacao_fn_tp.to_csv(caminho_erros, index=False, encoding="utf-8-sig")
if not amostra_fn.empty:
    amostra_fn.to_csv(caminho_fn_amostra, index=False, encoding="utf-8-sig")

for grupo, tabela in metricas_por_grupo_dict.items():
    caminho_grupo = TABLES_DIR / f"{prefixo}_por_grupo_{grupo}.csv"
    tabela.to_csv(caminho_grupo, index=False, encoding="utf-8-sig")

for ano, curva in curvas_por_ano.items():
    caminho_curva = TABLES_DIR / f"{prefixo}_curva_confiabilidade_{ano}.csv"
    curva.to_csv(caminho_curva, index=False, encoding="utf-8-sig")

try:
    limiar_final_value = float(limiar_final) if limiar_final is not None else 0.5
except (TypeError, ValueError):
    limiar_final_value = 0.5

cohort_kind_value = manifest.get("cohort_kind") if isinstance(manifest, dict) else None
cohort_kind_value = cohort_kind_value if cohort_kind_value is not None and cohort_kind_value != "" else "unknown"

modelo_value = finalista_escolhido if finalista_escolhido is not None else "unknown"

pred_test_df = pd.DataFrame({
    "y_true": pd.Series(y_test_arr),
    "y_prob": pd.Series(prob_test),
    "y_pred": pd.Series(y_pred_test),
    "limiar": float(limiar_final_value),
    "modelo": str(modelo_value),
    "cohort_kind": str(cohort_kind_value),
})
if COLUNA_ANO in teste.columns:
    pred_test_df[COLUNA_ANO] = teste[COLUNA_ANO].to_numpy()
pred_test_df.to_parquet(caminho_predicoes, index=False)

def _safe_float(value, default: float = 0.0) -> float:
    if value is None:
        return default
    try:
        return float(value)
    except (TypeError, ValueError):
        return default

cohort_kind_value = manifest.get("cohort_kind") if isinstance(manifest, dict) else None
cohort_kind_value = cohort_kind_value if cohort_kind_value not in (None, "") else "unknown"

dataset_path_value = manifest.get("dataset_path") if isinstance(manifest, dict) else None
dataset_sha256_value = manifest.get("dataset_sha256") if isinstance(manifest, dict) else None

metricas_agregadas = {}
for k, v in (dict(metricas_test or {}).items()):
    metricas_agregadas[str(k)] = _safe_float(v, default=0.0)

manifesto_c6 = {
    "cohort_kind": str(cohort_kind_value),
    "dataset_path": str(dataset_path_value) if dataset_path_value is not None else None,
    "dataset_sha256": dataset_sha256_value,
    "feature_columns": list(FEATURES) if FEATURES is not None else [],
    "target_column": ALVO,
    "partition_column": PARTICAO,
    "n_treino": int(len(treino)),
    "n_validacao": int(len(validacao)),
    "n_teste": int(len(teste)),
    "finalista_escolhido": finalista_escolhido if finalista_escolhido is not None else "unknown",
    "calibracao_ativa": bool(calib_ativa) if calib_ativa is not None else False,
    "calibracao_metodo": calib_metodo,
    "calibracao_fonte": calib_fonte,
    "limiar_final": _safe_float(limiar_final, default=0.5),
    "limiar_metrica": limiar_metrica,
    "g7_status": (C.cfg("gates", "G7_calibracao") or {}).get("status"),
    "g8_status": (C.cfg("gates", "G8_limiar") or {}).get("status"),
    "g9_status": (C.cfg("gates", "G9_holdout") or {}).get("status"),
    "test_opened_before": manifest.get("test_opened") if isinstance(manifest, dict) else None,
    "teste_aberto_em_utc": abertura_teste_utc,
    "metricas_agregadas": metricas_agregadas,
    "ic_bootstrap": ic_resultados,
    "erros_resumo": {
        "fn": int(len(fn)),
        "tp": int(len(tp)),
        "fp": int(len(fp)),
        "tn": int(len(tn)),
    },
    "grupos_avaliados": list(metricas_por_grupo_dict.keys()),
    "grupos_pulados": grupos_pulados,
    "artefatos": {
        "metricas": str(caminho_metricas),
        "matriz_confusao": str(caminho_matriz),
        "metricas_por_ano": str(caminho_por_ano),
        "calibracao_por_ano": str(caminho_calib_ano),
        "ic_bootstrap": str(caminho_ic) if caminho_ic.exists() else None,
        "analise_erros": str(caminho_erros) if caminho_erros.exists() else None,
        "fn_amostra": str(caminho_fn_amostra) if caminho_fn_amostra.exists() else None,
        "predicoes_teste": str(caminho_predicoes),
    },
    "observacoes": [
        "Teste aberto uma única vez, após G7/G8/G9 aprovados.",
        "Pipeline reconstruído deterministicamente a partir do supervised.yaml.",
        "Calibração ajustada na validação (fonte: validacao).",
        "IC por bootstrap para AP, AUC e Brier.",
        "Análise de erros com prioridade a falsos negativos.",
        "G9_holdout marcado como executado ao final desta execução.",
    ],
}

caminho_manifesto.write_text(
    json.dumps(manifesto_c6, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

assert pd.read_csv(caminho_metricas, encoding="utf-8-sig").shape[0] == 1
assert pd.read_parquet(caminho_predicoes).shape[0] == len(teste)

print("Métricas       :", caminho_metricas)
print("Matriz         :", caminho_matriz)
print("Métricas/ano   :", caminho_por_ano)
print("Calibração/ano :", caminho_calib_ano)
print("IC bootstrap   :", caminho_ic)
print("Análise erros  :", caminho_erros)
print("FN amostra     :", caminho_fn_amostra)
print("Predições      :", caminho_predicoes)
print("Manifesto      :", caminho_manifesto)

## 16. Registrar execução

In [ ]:
limiar_final_val = _safe_float(limiar_final, default=0.5)
run_metricas = {
    key: _safe_float(value, default=0.0)
    for key, value in (metricas_test or {}).items()
}

dataset_path_raw = manifest.get("dataset_path")
dataset_path = (
    Path(dataset_path_raw)
    if isinstance(dataset_path_raw, (str, os.PathLike))
    else ROOT_DIR
)

with Run(
    modelo="C6_validacao_final",
    parametros={
        "cohort_kind": manifest.get("cohort_kind"),
        "finalista_escolhido": finalista_escolhido,
        "calibracao_metodo": calib_metodo,
        "limiar_final": limiar_final_val,
        "limiar_metrica": limiar_metrica,
        "teste_aberto_em_utc": abertura_teste_utc,
        "test_opened_before": manifest.get("test_opened"),
        "n_acessos_antes": int(n_acessos_antes),
    },
    dataset={
        "n_treino": int(len(treino)),
        "n_validacao": int(len(validacao)),
        "n_teste": int(len(teste)),
    },
    features={
        "n_features": len(FEATURES),
        "usadas": list(FEATURES),
    },
    notas=(
        "C6 — abertura única do teste após G7/G8/G9 aprovados. "
        "Pipeline reconstruído via src.models.factory.build_finalista. "
        "IC por bootstrap e análise de erros inclusos."
    ),
    dataset_path=dataset_path,
    dataset_sha256=manifest.get("dataset_sha256"),
) as r:
    r.metrica(**run_metricas)

print("Run registrado em:", r.pasta)

## 17. Marcar `G9_holdout` como `executado`

Edição **cirúrgica** do `supervised.yaml`: só o bloco `G9_holdout` é tocado, preservando comentários do arquivo. O estado passa de `aprovado` → `executado` (terminal), grava o timestamp UTC e incrementa `n_acessos`.

Na próxima execução do C6 com o mesmo YAML, a checagem da seção 2 falhará — é assim que a disciplina de "abertura única" fica programaticamente imposta.

In [ ]:
def _marcar_g9_executado(root_dir: Path, timestamp_utc: str, n_acessos_antes: int) -> Path:
    """
    Marca G9_holdout como executado no supervised.yaml.

    Edição cirúrgica (texto cru), preservando comentários e formatação.
    Não usa yaml.dump para não remover os comentários do arquivo.
    """
    yaml_path = root_dir / "configs" / "supervised.yaml"
    texto = yaml_path.read_text(encoding="utf-8")

    linhas = texto.splitlines(keepends=True)
    i_inicio = None
    i_fim = None
    for i, linha in enumerate(linhas):
        if linha.rstrip() == "  G9_holdout:":
            i_inicio = i
            continue
        if (
            i_inicio is not None
            and linha.startswith("  ")
            and not linha.startswith("    ")
            and linha.strip()
        ):
            i_fim = i
            break

    if i_inicio is None:
        raise RuntimeError("Bloco `G9_holdout:` não encontrado em supervised.yaml.")
    if i_fim is None:
        i_fim = len(linhas)

    def _substituir(bloco, chave, valor_str):
        padrao = re.compile(rf"^(\s*){re.escape(chave)}:.*$")
        for j, l in enumerate(bloco):
            m = padrao.match(l)
            if m:
                bloco[j] = f"{m.group(1)}{chave}: {valor_str}\n"
                return True
        return False

    bloco = linhas[i_inicio:i_fim]
    ok_status = _substituir(bloco, "status", "executado")
    ok_data   = _substituir(bloco, "executado_em", f'"{timestamp_utc}"')
    ok_n      = _substituir(bloco, "n_acessos", str(n_acessos_antes + 1))

    if not (ok_status and ok_data and ok_n):
        raise RuntimeError(
            "Não foi possível atualizar todos os campos de G9_holdout "
            "(status, executado_em, n_acessos). Verifique o bloco no YAML."
        )

    linhas[i_inicio:i_fim] = bloco
    yaml_path.write_text("".join(linhas), encoding="utf-8")
    return yaml_path


yaml_path = _marcar_g9_executado(ROOT_DIR, abertura_teste_utc, n_acessos_antes)
print(f"G9_holdout marcado como executado em {yaml_path}")
print(f"executado_em: {abertura_teste_utc}")
print(f"n_acessos: {n_acessos_antes} → {n_acessos_antes + 1}")

## 18. Conclusão final

Depois desta célula, o teste está formalmente **fechado**. Qualquer alteração em modelo, features, calibração ou limiar exige um novo ciclo metodológico completo.

**Registrar em `docs/decisoes.md`:**

- finalista escolhido e hiperparâmetros congelados;
- método de calibração;
- limiar operacional e métrica;
- data/hora UTC da abertura do teste;
- métricas agregadas no teste **com IC bootstrap**;
- resumo da análise de erros (falsos negativos);
- principais limitações observadas na análise por ano e por grupo.

In [ ]:
checklist = {
    "gates_g7_g8_g9_aprovados_na_entrada": all(s == "aprovado" for s in status_gates.values()),
    "config_congelada_do_yaml": bool(finalista_escolhido) and bool(features_final),
    "teste_materializado": "teste" in globals(),
    "calibracao_ajustada_na_validacao": (not calib_ativa) or (calibrador is not None),
    "predicoes_teste_validas": bool(((prob_test >= 0) & (prob_test <= 1)).all()),
    "metricas_calculadas": bool(metricas_test),
    "ic_bootstrap_calculado": (not BOOT_ATIVO) or bool(ic_resultados),
    "analise_erros_executada": ERROS_ATIVA,
    "artefatos_salvos": caminho_manifesto.exists(),
    "run_registrado": r.pasta is not None,
    "g9_marcado_executado": True,
}

display(pd.DataFrame({
    "item": list(checklist.keys()),
    "concluido": list(checklist.values()),
}))

if not all(checklist.values()):
    pendencias = [k for k, v in checklist.items() if not v]
    raise RuntimeError(f"C6 terminou com pendências: {pendencias}")

print()
print("=" * 72)
print("C6 concluído. Teste aberto uma única vez.")
print("=" * 72)
print(f"Finalista: {finalista_escolhido}")
print(f"Calibração: {calib_metodo if calib_ativa else 'nenhuma'} (fonte: {calib_fonte})")
limiar_exibicao = _safe_float(limiar_final, default=float("nan"))
metrica_limiar = str(limiar_metrica) if limiar_metrica is not None else "desconhecida"
print(f"Limiar: {limiar_exibicao} ({metrica_limiar})")
print(f"AP no teste: {metricas_test['ap']:.4f}")
print(f"AUC no teste: {metricas_test['auc']:.4f}")
print(f"Brier no teste: {metricas_test['brier']:.4f}")
if ic_resultados:
    ap_ic = ic_resultados.get("ap", {})
    print(f"IC {BOOT_NIVEL:.0%} AP: [{ap_ic.get('ic_inferior', float('nan')):.4f}, {ap_ic.get('ic_superior', float('nan')):.4f}]")
print(f"Falsos negativos: {len(fn)} de {int(y_test_arr.sum())} óbitos no teste")
print()
print("G9_holdout marcado como executado em configs/supervised.yaml.")
print("Nada mais pode ser alterado. Registre as decisões em docs/decisoes.md.")